In [ ]:
with open('../projects/dataAnalyse-基于langgraph的多代理/company.txt', 'r', encoding="utf-8") as file:
    # 读取文件的全部内容
    content = file.read()
    print(content)

from langchain_core.documents import Document

documents = [Document(page_content=content)]

# GraphRAG Setup
from langchain_community.graphs import Neo4jGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain

# 创建图数据库示例
graph = Neo4jGraph(url='neo4j+s://e2f16b1a.databases.neo4j.io',  # 替换为自己的
                  username="neo4j",  # 替换为自己的
                  password="WvNNfKFQHwZurjccM0MFjz6SuKkuFNdT9y9R9E53CME", #替换为自己的
                  database="neo4j" # 替换为自己的
                  )

graph_llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini")

# 图转换器配置
graph_transformer = LLMGraphTransformer(
    llm=graph_llm,
    allowed_nodes=["公司", "产品", "技术", "市场", "活动", "合作伙伴"],    # 可以自定义节点
    allowed_relationships=["推出", "参与", "合作", "位于", "开发"],       # 可以自定义关系
)
graph_documents = graph_transformer.convert_to_graph_documents(documents)
graph.add_graph_documents(graph_documents)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

cypher_chain = GraphCypherQAChain.from_llm(
    graph=graph,
    cypher_llm=llm,
    qa_llm=llm,
    validate_cypher=True, # Validate relationship directions
    verbose=True,
    allow_dangerous_requests=True
)
cypher_chain.invoke("小米科技有限责任公司推出了哪些创新技术？")

In [ ]:
#传统rag
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_milvus import Milvus
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
splits = text_splitter.split_documents(documents)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large",)

vectorstore = Milvus.from_documents(
    documents=splits,
    collection_name="company_rag_milvus",
    embedding=embeddings,
    connection_args={
        "uri": "https://in03-33fcec99d39aeeb.serverless.gcp-us-west1.cloud.zilliz.com",
        "user": "db_33fcec99d39aeeb",
        "password": "Jw7+<}Yy*Cx!9(z*",
    }
)
prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks.
    Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know.
    Use three sentences maximum and keep the answer concise:
    Question: {question}
    Context: {context}
    Answer:
    """,
    input_variables=["question", "document"],
)
# 构建传统的RAG Chain
rag_chain = prompt | graph_llm | StrOutputParser()
# 构建检索器
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END

class AgentState(MessagesState):
    next: str

def vec_kg(state: AgentState):
    messages = state["messages"][-1]
    docs = retriever.invoke(messages.content)
    generation = rag_chain.invoke({"context": docs, "question": messages.content})
    final_response = [HumanMessage(content=generation, name="vec_kg")]   # 这里要添加名称
    return {"messages": final_response}

def graph_kg(state: AgentState):
    messages = state["messages"][-1]
    response = cypher_chain.invoke(messages.content)
    final_response = [HumanMessage(content=response["result"], name="graph_kg")]   # 这里要添加名称
    return {"messages": final_response}

def db_node(state: AgentState):
    result = db_agent.invoke(state)
    return {
        "messages": [
            HumanMessage(content=result["messages"][-1].content, name="sqler")
        ]
    }

def code_node(state: AgentState):
    result = code_agent.invoke(state)
    return {
        "messages": [HumanMessage(content=result["messages"][-1].content, name="coder")]
    }

def chat(state: AgentState):
    messages = state["messages"][-1]
    model_response = llm.invoke(messages.content)
    final_response = [HumanMessage(content=model_response.content, name="chatbot")]
    return {"messages": final_response}

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from typing import Literal
from typing_extensions import TypedDict

members = ["graph_kg", "vec_kg","chat", "coder", "sqler"]
options = members + ["FINISH"]

class Router(TypedDict):
    """Worker to route to next. If no workers needed, route to FINISH"""
    next: Literal[*options]

def supervisor(state: AgentState):
    system_prompt = (
        "You are a supervisor tasked with managing a conversation between the"
        f" following workers: {members}.\n\n"
        "Each worker has a specific role:\n"
        "- chat: Responds directly to user inputs using natural language.\n"
        "- graph_kg: Stores market and company information, built on a graph-based knowledge base, excels at answering broad and comprehensive questions.\n"
        "- vec_kg: Stores market and company information, constructed on a traditional semantic retrieval knowledge base, excels at answering detailed and fine-grained questions.\n"
        "Given the following user request, respond with the worker to act next."
        " Each worker will perform a task and respond with their results and status."
        " When finished, respond with FINISH."
    )

    messages = [{"role": "system", "content": system_prompt},] + state["messages"]
    response = llm.with_structured_output(Router).invoke(messages)
    return {"next": response["next"]}

builder = StateGraph(AgentState)

builder.add_node("supervisor", supervisor)
builder.add_node("chat", chat)
builder.add_node("coder", db_node)
builder.add_node("sqler", code_node)
builder.add_node("graph_kg", graph_kg)
builder.add_node("vec_kg", vec_kg)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor",lambda state: state['next'],{'chat': 'chat','coder': 'coder','sqler': 'sqler','FINISH': END})
for member in members:
    # 我们希望我们的工人在完成工作后总是向主管“汇报”
    builder.add_edge(member, "supervisor")

graph = builder.compile()

In [ ]:
for chunk in graph.stream({"messages": "都有哪些公司在我的数据库中。"}, stream_mode="values"):
    chunk["messages"][-1].pretty_print()